In [11]:
import pandas as pd
import numpy as np
import random

def generate_bottleneck_dataset(num_samples=10000):
    data = []
    
    for _ in range(num_samples):
        # 1. Simulate the current task
        burst_time = random.randint(1, 20)
        arrival_time = random.randint(0, 100)
        
        # 2. Simulate the state of the queue behind it
        queue_length = random.randint(0, 15)
        
        if queue_length > 0:
            queue_bursts = [random.randint(1, 10) for _ in range(queue_length)]
            avg_burst_in_queue = sum(queue_bursts) / queue_length
        else:
            avg_burst_in_queue = 0
            
        time_since_last_task = random.randint(0, 5)

        # 3. Calculate the "Perfect" Bottleneck Score (Our Target 'y')
        # This is the logic the AI will learn to predict.
        # Rule 1: If the task is huge and there are many short tasks waiting = HIGH Bottleneck
        # Rule 2: If the task is short, or no one is waiting = LOW Bottleneck
        
        if queue_length == 0:
            bottleneck_score = 0  # No bottleneck if no one is waiting
        else:
            # Impact = (Current Task Length) * (Number of tasks delayed)
            impact = burst_time * queue_length
            
            # Normalization factor: Are the waiting tasks shorter than the current one?
            ratio = burst_time / (avg_burst_in_queue + 1) 
            
            # Final calculated score
            bottleneck_score = impact * ratio
            
            # Scale down slightly for readability
            bottleneck_score = min(100, bottleneck_score / 10) 

        data.append({
            'burst_time': burst_time,
            'arrival_time': arrival_time,
            'queue_length': queue_length,
            'avg_burst_in_queue': avg_burst_in_queue,
            'time_since_last_task': time_since_last_task,
            'bottleneck_score': bottleneck_score
        })
        
    df = pd.DataFrame(data)
    return df

# Generate and save the dataset
df = generate_bottleneck_dataset(50000)
df.to_csv('bottleneck_training_data.csv', index=False)
print(f"Generated {len(df)} samples. Dataset looks like this:")
print(df.head())

Generated 50000 samples. Dataset looks like this:
   burst_time  arrival_time  queue_length  avg_burst_in_queue  \
0           1            23             5            5.400000   
1          20            27            13            5.384615   
2           7            93             2            2.000000   
3          14            83             4            4.500000   
4          15            92             5            5.600000   

   time_since_last_task  bottleneck_score  
0                     2          0.078125  
1                     4         81.445783  
2                     4          3.266667  
3                     1         14.254545  
4                     3         17.045455  
